# Weather API Integration

## 1. Project Setup

### 1.1 Import Libraries and Configuration

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import requests

import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "Notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    CLEAN_DATA_FILE,
    TARGET,
    MIGRAINE_FILTER,
    WEATHER_RAW_FEATURES,
    WEATHER_ENRICHED_DATA_FILE,
)

DATA_DIR = PROJECT_ROOT / "Data"


### 1.2 Load Data

In [2]:
# Load cleaned migraine diary data
diary = pd.read_csv(CLEAN_DATA_FILE)

In [3]:
# Restrict data to migraine episodes with known severity
model_df = diary.loc[
    (diary[MIGRAINE_FILTER] == 1)
    & diary[TARGET].notna()
].copy()

model_df = model_df.reset_index(drop=True)

print(f"Modeling observations: {model_df.shape[0]}")
print(model_df[TARGET].value_counts().sort_index())

Modeling observations: 302
Severity_VAS
Mild         55
Moderate    138
Severe      109
Name: count, dtype: int64


In [4]:
# Load dataset containing hospital information
hospital_df = pd.read_csv(
    DATA_DIR / "Intermediate" / "01_patients62.csv"
)

hospital_df.shape

(132, 185)

In [5]:
model_df.head()

,No.,Registration No.,Preventive Medication Use,Start Time,Index (VAS Style Score),Migraine (Y/N) Based on Premises from ICHD-3 Diagnosis,Internal Factor: Stress,Internal Factor: Oversleeping,Internal Factor: Sleep Deprivation,Internal Factor: Exercise,...,Moderate Exercise,Total Exercise,Headache-free Day,Severity_VAS,Start_Hour,Weekday,Month,Time_of_Day,Headache_Date,Headache_free_binary
0,2131,cmc-0027,1,2014-12-21 10:18:00,9.0,1.0,0,0,0,0,...,1,2,N,Severe,10.0,Sunday,12.0,Morning,2014-12-21,0
1,386,cmc-0032,0,2014-09-29 11:34:00,5.0,1.0,0,0,0,0,...,1,2,N,Moderate,11.0,Monday,9.0,Morning,2014-09-29,0
2,1842,cmc-0032,0,2014-11-29 11:20:00,6.0,1.0,0,0,0,0,...,1,2,N,Moderate,11.0,Saturday,11.0,Morning,2014-11-29,0
3,1589,cmc-0048,1,2014-11-24 20:50:00,4.0,1.0,0,0,0,0,...,1,1,N,Moderate,20.0,Monday,11.0,Evening,2014-11-24,0
4,3046,cmc-0077,1,2015-01-25 08:00:00,3.0,1.0,0,0,0,0,...,1,1,N,Mild,8.0,Sunday,1.0,Morning,2015-01-25,0


In [6]:
hospital_df.head()

,Unique number,Registration number,Study number,gender,years,hospital,height,Height unknown,weight,weight unknown,...,mediastinal exercise day,Total days of moderate exercise,shooting exercise time,Total moderate exercise time,Unnamed: 179,All in one day.1,mediastinal exercise day.1,Total days of moderate exercise.1,shooting exercise time.1,Total moderate exercise time.1
0,2.0,cmc-0001,M-001,female,20.0,Uijeongbu,155.0,NaN,56.0,NaN,...,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0
1,3.0,cmc-0002,M-002,female,38.0,Uijeongbu,155.0,NaN,53.0,NaN,...,0.0,6.0,0.0,190.0,NaN,1.0,0.0,1.0,0.0,30.0
2,5.0,cmc-0003,M-003,female,20.0,Uijeongbu,155.0,NaN,55.0,NaN,...,0.0,1.0,0.0,120.0,NaN,1.0,0.0,1.0,0.0,120.0
3,6.0,cmc-0004,M-004,female,37.0,Uijeongbu,164.0,NaN,52.0,NaN,...,0.0,4.0,0.0,280.0,NaN,1.0,0.0,1.0,0.0,40.0
4,8.0,cmc-0006,CM-001,female,46.0,Uijeongbu,157.0,NaN,52.0,NaN,...,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0


### 1.3 Create new weather dataframe that combines hospital information with cleaned data

In [7]:
# Add hospital information while preserving the original df
weather_df = model_df.merge(
    hospital_df[["Registration number", "hospital"]],
    how="left",
    left_on="Registration No.",
    right_on="Registration number"
)

# Remove the duplicate ID column from the hospital dataset
weather_df = weather_df.drop(columns="Registration number")

weather_df.head()

,No.,Registration No.,Preventive Medication Use,Start Time,Index (VAS Style Score),Migraine (Y/N) Based on Premises from ICHD-3 Diagnosis,Internal Factor: Stress,Internal Factor: Oversleeping,Internal Factor: Sleep Deprivation,Internal Factor: Exercise,...,Total Exercise,Headache-free Day,Severity_VAS,Start_Hour,Weekday,Month,Time_of_Day,Headache_Date,Headache_free_binary,hospital
0,2131,cmc-0027,1,2014-12-21 10:18:00,9.0,1.0,0,0,0,0,...,2,N,Severe,10.0,Sunday,12.0,Morning,2014-12-21,0,Uijeongbu
1,386,cmc-0032,0,2014-09-29 11:34:00,5.0,1.0,0,0,0,0,...,2,N,Moderate,11.0,Monday,9.0,Morning,2014-09-29,0,Uijeongbu
2,1842,cmc-0032,0,2014-11-29 11:20:00,6.0,1.0,0,0,0,0,...,2,N,Moderate,11.0,Saturday,11.0,Morning,2014-11-29,0,Uijeongbu
3,1589,cmc-0048,1,2014-11-24 20:50:00,4.0,1.0,0,0,0,0,...,1,N,Moderate,20.0,Monday,11.0,Evening,2014-11-24,0,Uijeongbu
4,3046,cmc-0077,1,2015-01-25 08:00:00,3.0,1.0,0,0,0,0,...,1,N,Mild,8.0,Sunday,1.0,Morning,2015-01-25,0,Uijeongbu


In [8]:
print(f"Original rows: {len(model_df)}")
print(f"Weather dataset rows: {len(weather_df)}")

Original rows: 302
Weather dataset rows: 302


In [9]:
weather_df["hospital"].value_counts(dropna=False)

hospital
Uijeongbu         192
Dongtan Hallim    110
Name: count, dtype: int64

In [10]:
unmatched_ids = weather_df.loc[
    weather_df["hospital"].isna(),
    "Registration No."
].unique()

print(f"Unmatched participant IDs: {len(unmatched_ids)}")
print(unmatched_ids)

Unmatched participant IDs: 0
<StringArray>
[]
Length: 0, dtype: str


In [11]:
# Assign hospital based on registration ID prefix
weather_df.loc[
    weather_df["hospital"].isna()
    & weather_df["Registration No."].str.upper().str.startswith("CMC"),
    "hospital"
] = "Uijeongbu"

weather_df["hospital"].value_counts(dropna=False)

hospital
Uijeongbu         192
Dongtan Hallim    110
Name: count, dtype: int64

One participant did not have an assigned hospital following the initial merge. Review of the source data showed that study registration IDs followed a site-specific naming convention, with CMC-prefixed IDs corresponding to Uijeongbu and DHA-prefixed IDs corresponding to Dongtan Hallym. The participant's hospital was therefore assigned to Uijeongbu based on the CMC registration prefix.

### 1.4 Inspect Date, Time, and Location Information

In [12]:
# Identify columns potentially related to date, time, participant, or location
candidate_columns = [
    col for col in weather_df.columns
    if any(
        term in col.lower()
        for term in [
            "date",
            "time",
            "registration",
            "id",
            "site",
            "hospital",
            "center",
            "location",
            "city",
        ]
    )
]

candidate_columns

['Registration No.', 'Start Time', 'Time_of_Day', 'Headache_Date', 'hospital']

In [13]:
# Inspect candidate columns
for col in candidate_columns:
    print(f"\n--- {col} ---")
    print(weather_df[col].head())
    print(f"Unique values: {weather_df[col].nunique(dropna=True)}")


--- Registration No. ---
0    cmc-0027
1    cmc-0032
2    cmc-0032
3    cmc-0048
4    cmc-0077
Name: Registration No., dtype: str
Unique values: 48

--- Start Time ---
0    2014-12-21 10:18:00
1    2014-09-29 11:34:00
2    2014-11-29 11:20:00
3    2014-11-24 20:50:00
4    2015-01-25 08:00:00
Name: Start Time, dtype: str
Unique values: 301

--- Time_of_Day ---
0    Morning
1    Morning
2    Morning
3    Evening
4    Morning
Name: Time_of_Day, dtype: str
Unique values: 4

--- Headache_Date ---
0    2014-12-21
1    2014-09-29
2    2014-11-29
3    2014-11-24
4    2015-01-25
Name: Headache_Date, dtype: str
Unique values: 156

--- hospital ---
0    Uijeongbu
1    Uijeongbu
2    Uijeongbu
3    Uijeongbu
4    Uijeongbu
Name: hospital, dtype: str
Unique values: 2


### 1.5 Prepare Temporal Variables

Convert migraine onset timestamps to datetime format and verify the temporal coverage of the diary data before retrieving historical weather observations.

In [14]:
# Convert temporal variables to datetime format
weather_df["Start Time"] = pd.to_datetime(
    weather_df["Start Time"], 
    errors="coerce"
    )

weather_df["Headache_Date"] = pd.to_datetime(
    weather_df["Headache_Date"], 
    errors="coerce"
    )

# Check temporal coverage
print(f"Earliest migraine onset: {weather_df['Start Time'].min()}")
print(f"Latest migraine onset: {weather_df['Start Time'].max()}")
print(f"Missing start times: {weather_df['Start Time'].isna().sum()}")

print(
    f"\nUnique headache dates: "
    f"{weather_df['Headache_Date'].nunique()}"
)

Earliest migraine onset: 2014-08-25 14:00:00
Latest migraine onset: 2015-04-05 09:00:00
Missing start times: 0

Unique headache dates: 156


## 2. Define Weather API Parameters

In [15]:
# Weather API configuration
API_URL = "https://archive-api.open-meteo.com/v1/archive"
TIMEZONE = "Asia/Seoul"

### 2.1 Define Hospital Locations

Historical weather data will be retrieved using the geographic coordinates of each participant's assigned study hospital. Hospital site was determined from the study registration information merged with the migraine diary data.

- **Uijeongbu St. Mary's Hospital:** Uijeongbu, Gyeonggi-do
- **Hallym University Dongtan Sacred Heart Hospital:** Hwaseong, Gyeonggi-do


In [16]:
# Define study hospital locations by latitude and longitude
hospital_locations = {
    "Uijeongbu": {
        "latitude": 37.75843,
        "longitude": 127.07772,
    },
    "Dongtan Hallim": {
        "latitude": 37.21651,
        "longitude": 127.07995,
    },
}

hospital_locations

{'Uijeongbu': {'latitude': 37.75843, 'longitude': 127.07772},
 'Dongtan Hallim': {'latitude': 37.21651, 'longitude': 127.07995}}

### 2.2 Define Historical Date Range

Determine the earliest and latest migraine dates required for historical weather retrieval.

In [17]:
# Define historical weather date range
start_date = weather_df["Start Time"].min().date().isoformat()
end_date = weather_df["Start Time"].max().date().isoformat()

print(f"Weather start date: {start_date}")
print(f"Weather end date: {end_date}")

Weather start date: 2014-08-25
Weather end date: 2015-04-05


### 2.3 Define Weather Variables

Specify the hourly environmental variables required for the migraine severity analysis.

In [18]:
# Define hourly weather variables
hourly_variables = [
    "temperature_2m",
    "relative_humidity_2m",
    "pressure_msl",
    "surface_pressure",
]

## 3. Retrieve Historical Weather Data

Historical hourly weather data are retrieved from the Open-Meteo Historical Weather API using the geographic coordinates of the two study hospitals. An initial test request is performed before retrieving weather data for the full study period.

### 3.1 Test API Request

Perform a short test request for the Uijeongbu study site to confirm that the API connection, geographic coordinates, temporal settings, and requested weather variables are functioning correctly.

In [19]:
# Select Uijeongbu for the test request
test_location = hospital_locations["Uijeongbu"]

# Define test API parameters
test_params = {
    "latitude": test_location["latitude"],
    "longitude": test_location["longitude"],
    "start_date": start_date,
    "end_date": (
        pd.to_datetime(start_date) + pd.Timedelta(days=2)
    ).date().isoformat(),
    "hourly": ",".join(hourly_variables),
    "timezone": TIMEZONE,
}

test_params

{'latitude': 37.75843,
 'longitude': 127.07772,
 'start_date': '2014-08-25',
 'end_date': '2014-08-27',
 'hourly': 'temperature_2m,relative_humidity_2m,pressure_msl,surface_pressure',
 'timezone': 'Asia/Seoul'}

In [20]:
# Send test request to Open-Meteo
test_response = requests.get(
    API_URL,
    params=test_params,
    timeout=30,
)

print(f"Status code: {test_response.status_code}")

Status code: 200


### 3.2 Inspect the API response

In [21]:
# Convert response to JSON
test_data = test_response.json()

test_data.keys()

dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly'])

In [22]:
print(test_data["timezone"])
print(test_data["hourly_units"])

Asia/Seoul
{'time': 'iso8601', 'temperature_2m': '°C', 'relative_humidity_2m': '%', 'pressure_msl': 'hPa', 'surface_pressure': 'hPa'}


In [23]:
test_data["hourly"].keys()

dict_keys(['time', 'temperature_2m', 'relative_humidity_2m', 'pressure_msl', 'surface_pressure'])

### 3.3 Convert API Response to DataFrame

In [24]:
# Convert hourly weather data to DataFrame
test_weather_df = pd.DataFrame(
    test_data["hourly"]
)

test_weather_df.head()

,time,temperature_2m,relative_humidity_2m,pressure_msl,surface_pressure
0,2014-08-25T00:00,22.9,92,1009.1,1001.9
1,2014-08-25T01:00,22.9,91,1008.9,1001.7
2,2014-08-25T02:00,22.8,92,1008.1,1000.9
3,2014-08-25T03:00,22.6,94,1008.1,1000.9
4,2014-08-25T04:00,22.3,92,1007.6,1000.4


In [25]:
# Convert weather timestamps to datetime
test_weather_df["time"] = pd.to_datetime(
    test_weather_df["time"]
)

test_weather_df.dtypes

time                    datetime64[us]
temperature_2m                 float64
relative_humidity_2m             int64
pressure_msl                   float64
surface_pressure               float64
dtype: object

In [26]:
test_weather_df.head(10)

,time,temperature_2m,relative_humidity_2m,pressure_msl,surface_pressure
0,2014-08-25 00:00:00,22.9,92,1009.1,1001.9
1,2014-08-25 01:00:00,22.9,91,1008.9,1001.7
2,2014-08-25 02:00:00,22.8,92,1008.1,1000.9
3,2014-08-25 03:00:00,22.6,94,1008.1,1000.9
4,2014-08-25 04:00:00,22.3,92,1007.6,1000.4
5,2014-08-25 05:00:00,22.4,90,1007.8,1000.6
6,2014-08-25 06:00:00,22.4,90,1007.6,1000.4
7,2014-08-25 07:00:00,22.5,91,1007.3,1000.1
8,2014-08-25 08:00:00,23.4,88,1007.8,1000.6
9,2014-08-25 09:00:00,24.9,82,1007.6,1000.5


### 3.4 Retrieve Full Historical Weather Data

Retrieve hourly historical weather observations for the complete study period at each hospital location and combine the site-specific results into a single weather dataset.

In [27]:
# Retrieve full historical weather data for both study hospitals
weather_frames = []

for hospital, location in hospital_locations.items():
    
    params = {
        "latitude": location["latitude"],
        "longitude": location["longitude"],
        "start_date": start_date,
        "end_date": end_date,
        "hourly": ",".join(hourly_variables),
        "timezone": TIMEZONE,
    }

    response = requests.get(
        API_URL,
        params=params,
        timeout=30,
    )

    response.raise_for_status()

    weather_data = response.json()

    site_weather = pd.DataFrame(
        weather_data["hourly"]
    )

    site_weather["time"] = pd.to_datetime(
        site_weather["time"]
    )

    site_weather["hospital"] = hospital

    weather_frames.append(site_weather)


# Combine weather data from both hospital locations
historical_weather_df = pd.concat(
    weather_frames,
    ignore_index=True
)

historical_weather_df.head()

,time,temperature_2m,relative_humidity_2m,pressure_msl,surface_pressure,hospital
0,2014-08-25 00:00:00,22.9,92,1009.1,1001.9,Uijeongbu
1,2014-08-25 01:00:00,22.9,91,1008.9,1001.7,Uijeongbu
2,2014-08-25 02:00:00,22.8,92,1008.1,1000.9,Uijeongbu
3,2014-08-25 03:00:00,22.6,94,1008.1,1000.9,Uijeongbu
4,2014-08-25 04:00:00,22.3,92,1007.6,1000.4,Uijeongbu


In [28]:
historical_weather_df.shape

(10752, 6)

In [29]:
historical_weather_df["hospital"].value_counts()

hospital
Uijeongbu         5376
Dongtan Hallim    5376
Name: count, dtype: int64

## 4. Validate Historical Weather Data

Validate the completeness, temporal coverage, and plausibility of the retrieved historical weather data before matching weather observations to individual migraine episodes.

### 4.1 Inspect Data Structure

Confirm that historical weather observations were retrieved for both study hospital locations and that each site contains the expected number of hourly observations.

In [30]:
# Inspect historical weather dataset
print(f"Dataset shape: {historical_weather_df.shape}")

display(historical_weather_df.head())

display(
    historical_weather_df["hospital"]
    .value_counts()
    .rename("Hourly Observations")
)

Dataset shape: (10752, 6)


,time,temperature_2m,relative_humidity_2m,pressure_msl,surface_pressure,hospital
0,2014-08-25 00:00:00,22.9,92,1009.1,1001.9,Uijeongbu
1,2014-08-25 01:00:00,22.9,91,1008.9,1001.7,Uijeongbu
2,2014-08-25 02:00:00,22.8,92,1008.1,1000.9,Uijeongbu
3,2014-08-25 03:00:00,22.6,94,1008.1,1000.9,Uijeongbu
4,2014-08-25 04:00:00,22.3,92,1007.6,1000.4,Uijeongbu


hospital
Uijeongbu         5376
Dongtan Hallim    5376
Name: Hourly Observations, dtype: int64

### 4.2 Check Missing Values

Evaluate the retrieved weather variables for missing observations that could affect subsequent matching with migraine episodes.

In [31]:
# Check missing values in historical weather data
weather_missing = (
    historical_weather_df
    .isna()
    .sum()
    .to_frame("Missing Values")
)

weather_missing["Percent Missing"] = (
    weather_missing["Missing Values"]
    / len(historical_weather_df)
    * 100
).round(2)

weather_missing

,Missing Values,Percent Missing
time,0,0.0
temperature_2m,0,0.0
relative_humidity_2m,0,0.0
pressure_msl,0,0.0
surface_pressure,0,0.0
hospital,0,0.0


### 4.3 Verify Temporal Coverage

Confirm that the retrieved historical weather data span the full temporal range required by the migraine diary dataset.

In [32]:
# Compare migraine and weather temporal coverage
temporal_coverage = pd.DataFrame({
    "Dataset": [
        "Migraine Diary",
        "Historical Weather"
    ],
    "Earliest Timestamp": [
        weather_df["Start Time"].min(),
        historical_weather_df["time"].min()
    ],
    "Latest Timestamp": [
        weather_df["Start Time"].max(),
        historical_weather_df["time"].max()
    ]
})

temporal_coverage

,Dataset,Earliest Timestamp,Latest Timestamp
0,Migraine Diary,2014-08-25 14:00:00,2015-04-05 09:00:00
1,Historical Weather,2014-08-25 00:00:00,2015-04-05 23:00:00


### 4.4 Validate Weather Variable Ranges

Review descriptive statistics for the retrieved weather variables to identify implausible values or unexpected ranges before integrating the weather data with the migraine diary.

In [33]:
# Summarize historical weather variables
weather_summary = (
    historical_weather_df[hourly_variables]
    .describe()
    .T
    .round(2)
)

weather_summary

,count,mean,std,min,25%,50%,75%,max
temperature_2m,10752.0,6.63,9.28,-16.7,-1.1,5.1,14.0,28.8
relative_humidity_2m,10752.0,66.46,19.96,16.0,50.0,67.0,85.0,100.0
pressure_msl,10752.0,1020.61,6.28,996.5,1016.4,1020.7,1024.8,1037.3
surface_pressure,10752.0,1012.79,6.07,989.3,1008.7,1012.9,1016.8,1029.2


In [34]:
# Compare weather variables across hospital locations
hospital_weather_summary = (
    historical_weather_df
    .groupby("hospital")[hourly_variables]
    .agg(["min", "mean", "max"])
    .round(2)
)

hospital_weather_summary

temperature_2m             relative_humidity_2m              \
                          min  mean   max                  min   mean  max   
hospital                                                                     
Dongtan Hallim          -10.4  6.75  28.5                   25  67.90  100   
Uijeongbu               -16.7  6.50  28.8                   16  65.03  100   

               pressure_msl                  surface_pressure                   
                        min     mean     max              min     mean     max  
hospital                                                                        
Dongtan Hallim        996.9  1020.63  1037.3            989.4  1012.69  1029.0  
Uijeongbu             996.5  1020.59  1037.3            989.3  1012.89  1029.2

### 4.5 Verify Hourly Continuity

Confirm that consecutive historical weather observations occur at one-hour intervals for each hospital site.

In [35]:
# Calculate time difference between consecutive observations
hourly_gaps = (
    historical_weather_df
    .sort_values(["hospital", "time"])
    .groupby("hospital")["time"]
    .diff()
)

hourly_gaps.value_counts()

time
0 days 01:00:00    10750
Name: count, dtype: int64

## 5. Merge Weather and Migraine Data

Match each migraine episode to the historical weather observation from the participant's study hospital at the corresponding hour of migraine onset.

### 5.1 Create Hourly Migraine Timestamps

Create an hourly matching timestamp from the exact migraine onset time while preserving the original `Start Time` variable.

In [36]:
# Match migraine onset to the nearest hourly weather observation
weather_df["Weather_Hour"] = (
    weather_df["Start Time"].dt.floor("h")
    + pd.to_timedelta(
        (weather_df["Start Time"].dt.minute >= 30).astype(int),
        unit="h"
    )
)

weather_df[
    ["Start Time", "Weather_Hour", "hospital"]
].head()

,Start Time,Weather_Hour,hospital
0,2014-12-21 10:18:00,2014-12-21 10:00:00,Uijeongbu
1,2014-09-29 11:34:00,2014-09-29 12:00:00,Uijeongbu
2,2014-11-29 11:20:00,2014-11-29 11:00:00,Uijeongbu
3,2014-11-24 20:50:00,2014-11-24 21:00:00,Uijeongbu
4,2015-01-25 08:00:00,2015-01-25 08:00:00,Uijeongbu


### 5.2 Calculate 24-Hour Weather Changes

Calculate changes in temperature and atmospheric pressure over the 24, 48, and 72 hours preceding each hourly weather observation. Changes are calculated separately for each hospital to preserve site-specific weather histories.

In [37]:
# Sort weather observations by hospital and time
historical_weather_df = historical_weather_df.sort_values(
    ["hospital", "time"]
).reset_index(drop=True)

# Calculate 24-hour weather changes within each hospital
historical_weather_df["relative_humidity_change_24h"] = (
    historical_weather_df.groupby("hospital")["relative_humidity_2m"]
    .diff(24)
)

historical_weather_df["temperature_change_24h"] = (
    historical_weather_df.groupby("hospital")["temperature_2m"]
    .diff(24)
)

historical_weather_df["pressure_msl_change_24h"] = (
    historical_weather_df.groupby("hospital")["pressure_msl"]
    .diff(24)
)

historical_weather_df["surface_pressure_change_24h"] = (
    historical_weather_df.groupby("hospital")["surface_pressure"]
    .diff(24)
)

# Calculate 48-hour weather changes within each hospital
historical_weather_df["relative_humidity_change_48h"] = (
    historical_weather_df.groupby("hospital")["relative_humidity_2m"]
    .diff(48)
)

historical_weather_df["temperature_change_48h"] = (
    historical_weather_df.groupby("hospital")["temperature_2m"]
    .diff(48)
)

historical_weather_df["pressure_msl_change_48h"] = (
    historical_weather_df.groupby("hospital")["pressure_msl"]
    .diff(48)
)

historical_weather_df["surface_pressure_change_48h"] = (
    historical_weather_df.groupby("hospital")["surface_pressure"]
    .diff(48)
)

# Calculate 72-hour weather changes within each hospital
historical_weather_df["relative_humidity_change_72h"] = (
    historical_weather_df.groupby("hospital")["relative_humidity_2m"]
    .diff(72)
)

historical_weather_df["temperature_change_72h"] = (
    historical_weather_df.groupby("hospital")["temperature_2m"]
    .diff(72)
)

historical_weather_df["pressure_msl_change_72h"] = (
    historical_weather_df.groupby("hospital")["pressure_msl"]
    .diff(72)
)

historical_weather_df["surface_pressure_change_72h"] = (
    historical_weather_df.groupby("hospital")["surface_pressure"]
    .diff(72)
)

In [38]:
historical_weather_df[
    WEATHER_RAW_FEATURES
].describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
temperature_2m,10752.0,6.63,9.28,-16.7,-1.1,5.1,14.0,28.8
relative_humidity_2m,10752.0,66.46,19.96,16.0,50.0,67.0,85.0,100.0
surface_pressure,10752.0,1012.79,6.07,989.3,1008.7,1012.9,1016.8,1029.2
pressure_msl,10752.0,1020.61,6.28,996.5,1016.4,1020.7,1024.8,1037.3


### 5.3 Prepare Historical Weather Data for Merge

Rename the weather timestamp to match the migraine dataset and retain the hospital identifier so that weather observations are matched by both location and time.

In [39]:
# Prepare weather data for merging
weather_lookup = historical_weather_df.rename(
    columns={"time": "Weather_Hour"}
).copy()

weather_lookup.head()

,Weather_Hour,temperature_2m,relative_humidity_2m,pressure_msl,surface_pressure,hospital,relative_humidity_change_24h,temperature_change_24h,pressure_msl_change_24h,surface_pressure_change_24h,relative_humidity_change_48h,temperature_change_48h,pressure_msl_change_48h,surface_pressure_change_48h,relative_humidity_change_72h,temperature_change_72h,pressure_msl_change_72h,surface_pressure_change_72h
0,2014-08-25 00:00:00,23.0,93,1009.0,1001.6,Dongtan Hallim,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2014-08-25 01:00:00,23.0,92,1008.5,1001.1,Dongtan Hallim,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2014-08-25 02:00:00,22.8,92,1007.6,1000.2,Dongtan Hallim,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2014-08-25 03:00:00,22.5,95,1007.6,1000.2,Dongtan Hallim,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2014-08-25 04:00:00,22.8,80,1007.2,999.8,Dongtan Hallim,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [40]:
# Confirm one weather observation per hospital and hour
duplicate_weather_keys = weather_lookup.duplicated(
    subset=["hospital", "Weather_Hour"]
).sum()

print(f"Duplicate hospital-hour combinations: {duplicate_weather_keys}")

Duplicate hospital-hour combinations: 0


### 5.4 Merge Weather with Migraine Records

Perform a left join so that all migraine diary records are retained while the corresponding hourly weather variables are added based on hospital and migraine onset hour.

In [41]:
# Merge hourly weather with migraine records
weather_enriched_df = weather_df.merge(
    weather_lookup,
    how="left",
    on=["hospital", "Weather_Hour"],
    validate="many_to_one"
)

weather_enriched_df.shape

(302, 56)

In [42]:
print(f"Rows before merge: {len(weather_df)}")
print(f"Rows after merge: {len(weather_enriched_df)}")

Rows before merge: 302
Rows after merge: 302


### 5.5 Validate Weather Match

Evaluate whether each migraine record successfully received the expected weather variables after the merge.

In [43]:
# Check missing weather values after merge
weather_match_missing = (
    weather_enriched_df[hourly_variables]
    .isna()
    .sum()
    .to_frame("Missing Values")
)

weather_match_missing["Percent Missing"] = (
    weather_match_missing["Missing Values"]
    / len(weather_enriched_df)
    * 100
).round(2)

weather_match_missing

,Missing Values,Percent Missing
temperature_2m,0,0.0
relative_humidity_2m,0,0.0
pressure_msl,0,0.0
surface_pressure,0,0.0


In [44]:
# Identify migraine records without a matched weather observation
unmatched_weather = weather_enriched_df[
    weather_enriched_df[hourly_variables].isna().all(axis=1)
]

print(f"Unmatched migraine records: {len(unmatched_weather)}")

Unmatched migraine records: 0


In [45]:
unmatched_weather[
    ["Registration No.", "Start Time", "Weather_Hour", "hospital"]
].head(20)

,Registration No.,Start Time,Weather_Hour,hospital


In [46]:
# Inspect merged migraine and weather data
weather_enriched_df[
    [
        "Registration No.",
        "Start Time",
        "Weather_Hour",
        "hospital",
        *hourly_variables,
    ]
].head(10)

,Registration No.,Start Time,Weather_Hour,hospital,temperature_2m,relative_humidity_2m,pressure_msl,surface_pressure
0,cmc-0027,2014-12-21 10:18:00,2014-12-21 10:00:00,Uijeongbu,-6.6,39,1024.3,1016.2
1,cmc-0032,2014-09-29 11:34:00,2014-09-29 12:00:00,Uijeongbu,20.2,88,1013.5,1006.2
2,cmc-0032,2014-11-29 11:20:00,2014-11-29 11:00:00,Uijeongbu,7.7,87,1023.2,1015.5
3,cmc-0048,2014-11-24 20:50:00,2014-11-24 21:00:00,Uijeongbu,6.3,80,1019.3,1011.6
4,cmc-0077,2015-01-25 08:00:00,2015-01-25 08:00:00,Uijeongbu,-0.9,85,1026.6,1018.7
5,dha-0010,2014-11-01 13:43:00,2014-11-01 14:00:00,Dongtan Hallim,18.6,65,1012.4,1004.8
6,dha-0041,2015-01-15 11:00:00,2015-01-15 11:00:00,Dongtan Hallim,4.0,70,1022.9,1014.9
7,cmc-0006,2014-10-13 00:01:00,2014-10-13 00:00:00,Uijeongbu,15.7,71,1013.5,1006.1
8,cmc-0006,2014-11-13 00:01:00,2014-11-13 00:00:00,Uijeongbu,-1.7,27,1021.2,1013.3
9,cmc-0009,2014-09-04 15:00:00,2014-09-04 15:00:00,Uijeongbu,25.0,65,1007.2,1000.1


### 5.6 Final Merge Validation and Sanity Checks

Perform final validation of the weather-enriched migraine dataset to confirm that all modeling observations were retained and successfully matched to historical weather data.

In [47]:
# Confirm all modeling observations were retained
print(f"Modeling observations: {len(model_df)}")
print(f"Weather-enriched observations: {len(weather_enriched_df)}")
print(f"Rows added or lost: {len(weather_enriched_df) - len(model_df)}")

Modeling observations: 302
Weather-enriched observations: 302
Rows added or lost: 0


In [48]:
# Check completeness of matched weather variables
weather_enriched_df[hourly_variables].isna().sum()

temperature_2m          0
relative_humidity_2m    0
pressure_msl            0
surface_pressure        0
dtype: int64

In [49]:
# Check completeness of 24-hour weather-change variables
weather_enriched_df[
    WEATHER_RAW_FEATURES
].isna().sum()

temperature_2m          0
relative_humidity_2m    0
surface_pressure        0
pressure_msl            0
dtype: int64

In [50]:
# Check missing hospital or weather-hour values
print(
    "Missing hospital:",
    weather_enriched_df["hospital"].isna().sum()
)

print(
    "Missing Weather_Hour:",
    weather_enriched_df["Weather_Hour"].isna().sum()
)

Missing hospital: 0
Missing Weather_Hour: 0


In [51]:
# Confirm weather matches across both hospital sites
weather_enriched_df.groupby("hospital").agg(
    observations=("Registration No.", "size"),
    earliest_onset=("Start Time", "min"),
    latest_onset=("Start Time", "max"),
)

,observations,earliest_onset,latest_onset
hospital,,,
Dongtan Hallim,110,2014-09-05 10:00:00,2015-03-20 15:40:00
Uijeongbu,192,2014-08-25 14:00:00,2015-04-05 09:00:00


## 6. Save Weather-Enriched Dataset

Save the validated weather-enriched migraine dataset for subsequent exploratory data analysis, feature engineering, and model development.

In [52]:
# Save weather-enriched migraine dataset
weather_enriched_df.to_csv(
    WEATHER_ENRICHED_DATA_FILE,
    index=False
)

print(f"Saved weather-enriched data to:")
print(WEATHER_ENRICHED_DATA_FILE)

Saved weather-enriched data to:
C:\Users\rebec\Documents\Thesis\Data\Processed\migraine_weather_enriched.csv


In [53]:
# Reload saved dataset to verify successful export
saved_weather_df = pd.read_csv(
    WEATHER_ENRICHED_DATA_FILE
)

print(f"Saved dataset shape: {saved_weather_df.shape}")
print(f"Expected dataset shape: {weather_enriched_df.shape}")

assert saved_weather_df.shape == weather_enriched_df.shape

print("Weather-enriched dataset saved successfully.")

Saved dataset shape: (302, 56)
Expected dataset shape: (302, 56)
Weather-enriched dataset saved successfully.
